<a href="https://colab.research.google.com/github/Shanta6039/finance-assistant-/blob/main/finance_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

   # Student Budget Checker
   ISYS2001 Assessment 2 — Programming Project
   Author: Shanta

## Step 1: Understand the problem

**1. Who is my user?**
My target users are university students aged 18–25 who have limited income from part-time jobs, family support or scholarships. They often have to manage expenses such as rent, groceries, transport, tuition fees and entertainment.

**2. What is their problem?**
Students often struggle to manage their money because they do not track their daily spending or follow a budget. As a result, they may run out of money before the end of the month without knowing where it went.

**3. Why does it matter?**
Poor money management can cause financial stress, unpaid bills and difficulty covering essential expenses. It can also prevent students from saving money and force them to borrow from friends or family.

**4. How will my app help?**
My Student Budget Checker lets students upload a CSV file of their transactions. It adds up their spending in each category, compares it with their monthly budget, and warns them when they spend more than their budget. Students can also chat with an AI assistant that gives personalised saving tips based on their real spending numbers.

## Step 2: Inputs and outputs

**Inputs (what goes into my app)**
- **Transactions CSV file:** a file of the student's spending with the columns date, description, category and amount (e.g. groceries, transport, entertainment).
- **Monthly budget per category:** how much the student plans to spend in each category each month.
- **Chat questions:** questions the student types to the AI assistant (e.g. "Where am I spending too much?").

**Outputs (what comes out of my app)**
- **Spending summary:** total spending in each category and overall.
- **Budget status:** whether each category is within or over budget.
- **Remaining balance:** how much money is left in each category (budget − spent), or how much the student went over.
- **Budget alerts:** warnings when the student has gone over budget in a category.
- **Saving tips:** personalised suggestions from the AI assistant, based on the student's real spending numbers.

**What could go wrong with the inputs**
- No file is uploaded, or the file is empty.
- The CSV is missing a column or has an amount that is not a number.
- A budget is negative, zero or not a number.
- A category in the CSV has no budget set.
- The AI service (Gemini) is busy or not responding.

In [ ]:
# Install the Gemini library (only needed once per session)
!pip install -q google-genai

import time                           # lets us wait between tries
from google.colab import userdata     # reads Colab Secrets
from google import genai              # Google's Gemini library
from google.genai import errors       # Gemini's error types

# Get the key from Secrets and connect to Gemini
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# Models to try, in order. If one is busy, we try the next.
models_to_try = ["gemini-flash-latest", "gemini-flash-lite-latest"]

def ask_gemini(question):
    """Send a question to Gemini. Retry if busy, and never crash."""
    for model in models_to_try:
        for attempt in range(1, 4):          # up to 3 tries per model
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=question
                )
                print(f"✅ Success using {model} (try {attempt})")
                return response.text
            except errors.ServerError:
                # 503: Google is busy. Wait, then try again.
                print(f"⏳ {model} is busy (try {attempt}), waiting 5 seconds...")
                time.sleep(5)
            except errors.ClientError as e:
                # 4xx: a problem on our side, e.g. wrong key or unknown model
                print(f"❌ Problem with {model}: {e}")
                break                         # skip to the next model
    return "Sorry, the assistant is busy right now. Please try again in a few minutes."

# Test it
print(ask_gemini("Say hello to Shanta in one short sentence."))

⏳ gemini-flash-latest is busy (try 1), waiting 5 seconds...
⏳ gemini-flash-latest is busy (try 2), waiting 5 seconds...
⏳ gemini-flash-latest is busy (try 3), waiting 5 seconds...
✅ Success using gemini-flash-lite-latest (try 1)
Hello, Shanta, I hope you're having a wonderful day!
